# Gold Common
공용 설정/유틸. Silver 공용 노트북(`00_silver_common`)을 그대로 상속하고, Gold 전용 테이블 이름/임계값/헬퍼만 추가한다.
Fabric은 `%run <notebook_display_name>`으로 워크스페이스 내 다른 노트북을 이름으로 참조한다 — 로컬 리포에서는 `fabric_notebooks/silver/00_silver_common.ipynb`와 `fabric_notebooks/gold/00_gold_common.ipynb`가 다른 폴더에 있지만, Fabric 워크스페이스에 두 노트북을 모두 올리면 이름으로 정상 참조된다.

### Notebook Parameters

In [ ]:
# PARAMETERS CELL
run_mode = "DAILY"        # INITIAL | DAILY | EVENT | BACKFILL | REPLAY
source = "gold"
snapshot_date = None       # "YYYY-MM-DD", None이면 오늘(UTC)
target_cve_ids = []        # EVENT / BACKFILL 대상 CVE
target_ghsa_ids = []       # 오늘 변경된 GHSA 집합을 파이프라인이 명시적으로 넘길 때 사용 (없으면 batch_id로 판별)
batch_id = None            # Pipeline이 발급해 GHSA/EPSS 소스 배치와 동일하게 맞춰 전달해야 함
is_backfill = False


In [ ]:
%run 00_silver_common

### Gold 테이블 이름

In [ ]:
GOLD_SCHEMA = "gold"

TBL.update({
    "gold_vuln_package": f"{GOLD_SCHEMA}.gold_vuln_package",
    "gold_vuln_daily": f"{GOLD_SCHEMA}.gold_vuln_daily",
    "publish_vulnerability_package": f"{GOLD_SCHEMA}.publish_vulnerability_package",
    "publish_risk_current": f"{GOLD_SCHEMA}.publish_risk_current",
    "publish_epss_trend_30d": f"{GOLD_SCHEMA}.publish_epss_trend_30d",
    "publish_risk_events": f"{GOLD_SCHEMA}.publish_risk_events",
    "ext_priority_score": f"{GOLD_SCHEMA}.ext_priority_score",  # DS팀이 채워주는 외부 입력 (03_Gold_Schema.md §3.10 "일정 분리")
    "meta_circuit_breaker_run": f"{GOLD_SCHEMA}.meta_circuit_breaker_run",
})


### Circuit Breaker 임계값
03_Gold_Schema.md §3.11 "Gold Publish 서킷 브레이커" 표에 명시된, 이미 실측/확정된 값을 그대로 재사용한다(새로 지어내지 않음).

In [ ]:
JOIN_MATCH_RATE_THRESHOLD = 0.90      # CVE -> EPSS 매치율 (기존 silver_epss_build 기준 재사용)
DUPLICATE_RATE_MAX = 0                # Business Key 중복 허용 건수
SCHEMA_DRIFT_MAX_COLUMNS = 0          # 신규/소실 컬럼 허용 건수


### Gold 전용 헬퍼
`replace_by_key`: `silver_advisory_ghsa_affected`와 동일한 "부모 키 단위 전체 교체" 패턴을 `gold_vuln_package`(부모 키 = `ghsa_id`)에도 그대로 적용한다 — 이 테이블은 valid_from/valid_to 이력을 두지 않기 때문이다(03_Gold_Schema.md §3.7).

In [ ]:
def replace_by_key(new_df: DataFrame, table_name: str, key_col: str, key_values: list):
    if table_exists(table_name):
        if key_values:
            t = DeltaTable.forName(spark, table_name)
            t.delete(F.col(key_col).isin(key_values))
        write_append(new_df, table_name)
    else:
        write_overwrite(new_df, table_name)


In [ ]:
def kev_asof_join(df: DataFrame, cve_col: str = "cve_id", date_col: str = "date_key") -> DataFrame:
    """df[date_col] 시점 기준 as-of KEV 상태를 붙인다: valid_from <= date_col < valid_to(또는 아직 열려있음).
    KEV 신호가 EPSS 값을 바꾸지 않는다는 원칙(05_Job_Orchestration.md §6.4)에 따라 EPSS 컬럼은 건드리지 않는다."""
    base_cols = df.columns
    if not table_exists(TBL["kev"]):
        return (
            df.withColumn("kev_listed", F.lit(False))
              .withColumn("kev_date_added", F.lit(None).cast(StringType()))
              .withColumn("kev_required_action", F.lit(None).cast(StringType()))
              .withColumn("has_kev", F.lit(False))
              .withColumn("days_since_kev_added", F.lit(None).cast(LongType()))
        )
    kev = spark.table(TBL["kev"])
    joined = (
        df.alias("d")
        .join(
            kev.alias("k"),
            (F.col(f"d.{cve_col}") == F.col("k.cve_id"))
            & (F.col("k.valid_from") <= F.col(f"d.{date_col}"))
            & (F.col("k.valid_to").isNull() | (F.col("k.valid_to") > F.col(f"d.{date_col}"))),
            "left",
        )
        .select(
            *[F.col(f"d.{c}") for c in base_cols],
            F.coalesce(F.col("k.kev_listed"), F.lit(False)).alias("kev_listed"),
            F.col("k.date_added").alias("kev_date_added"),
            F.col("k.required_action").alias("kev_required_action"),
        )
    )
    return (
        joined
        .withColumn("has_kev", F.col("kev_listed"))
        .withColumn(
            "days_since_kev_added",
            F.when(
                F.col("kev_date_added").isNotNull(),
                F.datediff(F.to_date(F.col(date_col)), F.to_date(F.substring(F.col("kev_date_added"), 1, 10))),
            ),
        )
    )


In [ ]:
def alias_ghsa_ids_join(df: DataFrame, cve_col: str = "cve_id") -> DataFrame:
    alias = spark.table(TBL["vuln_alias"]).filter(F.col("id_type") == "CVE")
    agg = alias.groupBy(F.col("id_value").alias(cve_col)).agg(
        F.sort_array(F.collect_set("canonical_key")).alias("ghsa_ids")
    )
    return df.join(agg, on=cve_col, how="left").withColumn(
        "ghsa_ids", F.coalesce(F.col("ghsa_ids"), F.array().cast("array<string>"))
    )

def systemic_join(df: DataFrame, date_col: str = "date_key") -> DataFrame:
    """silver_epss_distribution이 그 날짜에 아직 없으면 LEFT JOIN 결과가 자연스럽게 NULL이 되어
    is_systemic_day의 3상태(NULL=미판정) 원칙이 별도 처리 없이 지켜진다."""
    if not table_exists(TBL["epss_distribution"]):
        return (
            df.withColumn("is_systemic_day", F.lit(None).cast(BooleanType()))
              .withColumn("systemic_reason", F.lit(None).cast(StringType()))
        )
    dist = spark.table(TBL["epss_distribution"]).select(
        F.col("snapshot_date").alias(date_col),
        F.col("is_systemic").alias("is_systemic_day"),
        F.col("systemic_reason"),
    )
    return df.join(dist, on=date_col, how="left")
